## RAG Pipeline - Data Ingestion to Vector DB

In [1]:
import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import sys
print(sys.executable)

/var/folders/3d/n1gdt7t127l2b51qkg822cpc0000gn/T/ipykernel_97635/735092744.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


/Users/jatindergill/Desktop/Internship/AI-Course/Learn-Langgraph/.venv/bin/python


### Process and Load all PDF's

In [4]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory: str):
    """Process all the pdfs in a directory"""
    
    # list to store all loaded docs
    all_docs = []

    pdf_dir = Path(pdf_directory)

    #Find all PDF Recursively
    #.glob is to find file names matcing the pdf pattern, **/ means “also look through subfolders,
    # so it finds PDFs inside data/pdf and its nested folders.
    pdf_files = list(pdf_dir.glob("**/*.pdf")) # list -> [ Path("/dir/example.pdf") , Path("/dir/example2.pdf") ]
    print(f"Found {len(pdf_files)} PDF files to be processed")

    for file in pdf_files:
        print(f"\n Processing{file.name}") #.name is a property of Path object -> example.pdf from the /dir/example.pdf path 

        try: 
            loader = PyPDFLoader(str(file))
            documents = loader.load() #PDFLoader.load() returns a list of LangChain Document objects, usually one Document per PDF page.

            #this is the way to extract any suffix, we could have just use the string "pdf" as well in metadata
            fileType = file.suffix
            fileTypeStr = fileType.lstrip(".")

            # Add Source Info to Metadata
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = fileTypeStr

            all_docs.extend(documents) # eg: ["file1"] and ["file2", "file3"] becomes  ["file1", "file2", "file3"] if we use .append it becomes ["file1", "file2", "file3"]
            print(f" ✅ Loaded {len(documents)} Pages")

        except Exception as e:
            print(f" ❌ Error: {e}")
            
    print(f"\n Total Documents Loaded {len(all_docs)}")
    return all_docs

all_pdf_documents = process_all_pdfs("data")

    

Found 2 PDF files to be processed

 Processingnorthstar_support_policy.pdf
 ✅ Loaded 1 Pages

 Processingvector_search_field_guide.pdf
 ✅ Loaded 1 Pages

 Total Documents Loaded 2


### Split the Loaded Docuemnts into Chunks

In [5]:
# text splitting in chunks 


def chunk_documents(loaded_docs, chunk_size = 1000, chunk_overlap=200):
    """Splitting Text in smaller chunks for better RAG Performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len, # length_function=len tells it how to measure a chunk’s size. Python’s len() counts characters in a string:

        #separators tells the splitter where it should try to break the text, in order of preference:
        # "\n\n": between paragraphs
        #"\n": between lines
        #" ": between words
        #"": between individual characters, as a last resort
        separators=["\n\n", "\n", " ", ""] 
    )

    split_docs = text_splitter.split_documents(loaded_docs)
    print(f"Splitted Docs: {split_docs}")
    print(f"Spliting {len(loaded_docs)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example Chunk:")
        print(f"Content: {split_docs[0].page_content[:100]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [52]:
chunks = chunk_documents(all_pdf_documents)
chunks

Splitted Docs: [Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-09T01:43:02-07:00', 'author': 'RAG Learning Samples', 'keywords': '', 'moddate': '2026-11-09T01:43:02-07:00', 'subject': '(unspecified)', 'title': 'Northstar Learning: Support Policy', 'trapped': '/False', 'source': 'data/pdf/northstar_support_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'northstar_support_policy.pdf', 'file_type': 'pdf'}, page_content='RAG learning sample | Fictional training material\nPage 1\nSAMPLE DOCUMENT 01 / CUSTOMER SUPPORT POLICY\nNorthstar Learning: Support Policy\nNorthstar Learning is a fictional online course provider. This document is a sample policy for practicing PDF loading,\nchunking, metadata capture, and retrieval. The dates, amounts, and contact details below are invented for this exercise.\nSupport hours and response times\nThe support desk is open Monday through Friday, 9:00 a.m

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-09T01:43:02-07:00', 'author': 'RAG Learning Samples', 'keywords': '', 'moddate': '2026-11-09T01:43:02-07:00', 'subject': '(unspecified)', 'title': 'Northstar Learning: Support Policy', 'trapped': '/False', 'source': 'data/pdf/northstar_support_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'northstar_support_policy.pdf', 'file_type': 'pdf'}, page_content='RAG learning sample | Fictional training material\nPage 1\nSAMPLE DOCUMENT 01 / CUSTOMER SUPPORT POLICY\nNorthstar Learning: Support Policy\nNorthstar Learning is a fictional online course provider. This document is a sample policy for practicing PDF loading,\nchunking, metadata capture, and retrieval. The dates, amounts, and contact details below are invented for this exercise.\nSupport hours and response times\nThe support desk is open Monday through Friday, 9:00 a.m. to 5:00 p.m. 

### Embedding

In [6]:
import numpy as np  # Work with arrays and numerical vectors
from sentence_transformers import SentenceTransformer  # Convert text into embeddings
import chromadb  # Store and search embeddings in a vector database
from chromadb.config import Settings  # Configure ChromaDB
import uuid  # Generate unique IDs for stored documents
from typing import List, Dict, Any, Tuple  # Describe expected data types
from sklearn.metrics.pairwise import cosine_similarity  # Measure similarity between vectors

/Users/jatindergill/Desktop/Internship/AI-Course/Learn-Langgraph/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 01:41:39.703000 97635 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding Manager


        Args: 
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()


    # _load_model is protected function, which is a class scope function as it starts with _
    def _load_model(self):
        """Load SentenceTransformer Model"""
        try: 
            print(f"Loading embedding Model {self.model_name}")
            self.model = SentenceTransformer(self.model_name) #initalize the model that convert text into embeddings
            print(f"Model Loaded Sucessfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e: 
            print(f"Error loading Model {self.model_name}: {e}")
            raise 

    
    #NumPy is a Python library for working with numbers efficiently, eg: numbers = np.array([1, 2, 3]) --> numbers * 2  # array([2, 4, 6])
    # With a regular list, [1, 2, 3] * 2 repeats the list instead: [1, 2, 3, 1, 2, 3].
    #ndarray stands for N-dimensional array. It’s the type of object NumPy creates with np.array():
    #“N-dimensional” means it can hold data in different shapes:
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate Embeddings for a list of texts
        
        
        Args: 
            texts: List of Texts strings to embed

        Returns: 
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        #.encode is the method of SentenceTransformer which is the model, example: 
        # embeddings = model.encode([
            # "What is RAG?",
            #   "Python has dictionaries.",
            #])

            #print(embeddings.shape)  # for example, (2, 384) --> is the shape of an array containing embeddings:
            #2 = two input texts, so there are two embeddings
            #384 = each embedding contains 384 numbers

        embeddings = self.model.encode(texts, show_progress_bar=True) 
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings



In [14]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding Model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6677.90it/s]


Model Loaded Sucessfully. Embedding dimention: 384


/var/folders/3d/n1gdt7t127l2b51qkg822cpc0000gn/T/ipykernel_97635/658167637.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Sucessfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}")


### Vector Store